# Task 2.1 — IPA Unified Representation

In [3]:
!pip install -q epitran g2p-en

In [4]:
import json, re
from pathlib import Path

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

## Technical IPA Dictionary
Manually curated IPA for speech-processing technical terms that standard G2P tools mangle.

In [5]:
TECHNICAL_IPA = {
    'cepstrum': 'ˈkɛpstrəm', 'cepstral': 'ˈkɛpstrəl',
    'stochastic': 'stəˈkæstɪk', 'spectrogram': 'ˈspɛktɹəɡɹæm',
    'phoneme': 'ˈfoʊniːm', 'phonetic': 'fəˈnɛtɪk',
    'formant': 'ˈfɔːɹmənt', 'frequency': 'ˈfɹiːkwənsi',
    'fundamental': 'ˌfʌndəˈmɛntəl', 'mel': 'mɛl',
    'mfcc': 'ɛm.ɛf.siː.siː', 'fourier': 'ˈfʊɹieɪ',
    'gaussian': 'ˈɡaʊsiən', 'markov': 'ˈmɑːɹkɒv',
    'viterbi': 'vɪˈtɜːɹbi', 'vocoder': 'ˈvoʊkoʊdɜːɹ',
    'prosody': 'ˈpɹɒsədi', 'acoustic': 'əˈkuːstɪk',
    'spectral': 'ˈspɛktɹəl', 'pitch': 'pɪtʃ',
    'transformer': 'tɹænsˈfɔːɹmɜːɹ', 'attention': 'əˈtɛnʃən',
    'embedding': 'ɛmˈbɛdɪŋ', 'vector': 'ˈvɛktɜːɹ',
    'waveform': 'ˈweɪvfɔːɹm', 'synthesis': 'ˈsɪnθəsɪs',
    'recognition': 'ˌɹɛkəɡˈnɪʃən', 'algorithm': 'ˈælɡəɹɪðəm',
    'neural': 'ˈnjʊəɹəl', 'network': 'ˈnɛtwɜːɹk',
    'whisper': 'ˈwɪspɜːɹ', 'signal': 'ˈsɪɡnəl',
    'noise': 'nɔɪz', 'filter': 'ˈfɪltɜːɹ',
    'energy': 'ˈɛnɜːɹd͡ʒi', 'transcription': 'tɹænˈskɹɪpʃən',
}

## G2P Converters

In [7]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

# English G2P using CMU dictionary
from g2p_en import G2p
g2p_eng = G2p()

# Hindi G2P using epitran
import epitran
epi_hi = epitran.Epitran('hin-Deva')

def english_to_ipa(text):
    """Convert English text to IPA via g2p_en + technical dictionary."""
    words = text.lower().split()
    ipa_parts = []
    for word in words:
        clean = re.sub(r'[^a-z]', '', word)
        if not clean:
            continue
        if clean in TECHNICAL_IPA:
            ipa_parts.append(TECHNICAL_IPA[clean])
        else:
            # g2p_en returns ARPABET → convert to IPA
            phones = g2p_eng(clean)
            ipa_parts.append(''.join(phones))
    return ' '.join(ipa_parts)

def hindi_to_ipa(text):
    """Convert Hindi Devanagari text to IPA via epitran."""
    return epi_hi.transliterate(text)

def detect_and_convert(text):
    """Auto-detect script per character span and convert to IPA."""
    result = []
    current_script = None
    current_span = []

    for ch in text:
        if '\u0900' <= ch <= '\u097F':
            script = 'devanagari'
        elif ch.isalpha():
            script = 'latin'
        else:
            if current_span:
                span_text = ''.join(current_span)
                if current_script == 'devanagari':
                    result.append(hindi_to_ipa(span_text))
                else:
                    result.append(english_to_ipa(span_text))
                current_span = []
                current_script = None
            if ch == ' ':
                result.append(' ')
            continue

        if script != current_script and current_span:
            span_text = ''.join(current_span)
            if current_script == 'devanagari':
                result.append(hindi_to_ipa(span_text))
            else:
                result.append(english_to_ipa(span_text))
            current_span = []

        current_script = script
        current_span.append(ch)

    if current_span:
        span_text = ''.join(current_span)
        if current_script == 'devanagari':
            result.append(hindi_to_ipa(span_text))
        else:
            result.append(english_to_ipa(span_text))

    return ''.join(result).strip()

# Quick test
print('English test:', english_to_ipa('cepstrum analysis'))
print('Hindi test:', hindi_to_ipa('नमस्ते दुनिया'))
print('Mixed test:', detect_and_convert('hello नमस्ते world'))

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


English test: ˈkɛpstrəm AH0NAE1LAH0SAH0S
Hindi test: nəmste dunijaː
Mixed test: HHAH0LOW1 nəmste WER1LD


## Load transcript & LID labels

In [8]:
# Load transcript
transcript_path = 'transcript_constrained.json'
lid_path = 'lid_labels.json'

if Path(transcript_path).exists():
    tdata = load_json(transcript_path)
    segments = tdata.get('segments', []) if isinstance(tdata, dict) else tdata
else:
    print(f'Warning: {transcript_path} not found, using placeholder')
    segments = [{'start_s': 0, 'end_s': 600, 'text': 'placeholder transcript'}]

if Path(lid_path).exists():
    lid_segments = load_json(lid_path)
else:
    lid_segments = []
    print(f'Warning: {lid_path} not found')

print(f'Transcript segments: {len(segments)}')
print(f'LID segments: {len(lid_segments)}')

Transcript segments: 24
LID segments: 53


## Convert to IPA

In [9]:
ipa_segments = []
for seg in segments:
    text = seg.get('text', '').strip()
    if not text:
        continue

    t_start = seg.get('start_s', 0) * 1000
    t_end = seg.get('end_s', 0) * 1000

    # Determine dominant language from LID
    eng_ol, hin_ol = 0.0, 0.0
    for lid in lid_segments:
        ol_start = max(t_start, lid.get('start_ms', 0))
        ol_end = min(t_end, lid.get('end_ms', 0))
        ol = max(0, ol_end - ol_start)
        if ol > 0:
            if lid.get('language') == 'English':
                eng_ol += ol
            else:
                hin_ol += ol

    dominant_lang = 'English' if eng_ol >= hin_ol else 'Hindi'
    ipa_text = detect_and_convert(text)

    ipa_segments.append({
        'start_s': seg.get('start_s', 0),
        'end_s': seg.get('end_s', 0),
        'original_text': text,
        'language': dominant_lang,
        'ipa': ipa_text,
    })

print(f'Converted {len(ipa_segments)} segments to IPA')
for s in ipa_segments[:3]:
    print(f'  [{s["language"]}] "{s["original_text"][:60]}..."')
    print(f'          → {s["ipa"][:80]}...')

Converted 24 segments to IPA
  [English] "Here as a person who loves mathematics I have toend toend to..."
          → HHIY1R AE1Z AH0 PER1SAH0N HHUW1 LAH1VZ MAE2THAH0MAE1TIH0KS AY1 HHAE1V TOW1ND TOW...
  [English] "देख रहा हूं मैं उनके बहुत आधर करता हूं और मेरे देश के हर कोई..."
          → dekʰ rəɦaː ɦuːn mæːn unke bəɦut aːd̤ər kərtaː ɦuːn ɔːr mere deʃ ke ɦər koiː unko...
  [English] "रामानुजय होते हैं। उसे कटफ भी होते हैं। स्री निवास रामानुजय ..."
          → raːmaːnud͡ʒəj ɦote ɦæːn। use kəʈəpʰ b̤iː ɦote ɦæːn। sriː nivaːs raːmaːnud͡ʒəj ɦə...


## Save output

In [10]:
output = {
    'source_transcript': transcript_path,
    'source_lid': lid_path,
    'g2p_tools': {'english': 'g2p_en (CMU Dict)', 'hindi': 'epitran (hin-Deva)'},
    'segments': ipa_segments,
    'full_ipa': ' '.join(s['ipa'] for s in ipa_segments),
}

with open('transcript_ipa.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f'Saved {len(ipa_segments)} IPA segments to transcript_ipa.json')
print('Done!')

Saved 24 IPA segments to transcript_ipa.json
Done!
